## Create water masks from RTC or NRB data in an Open Data Cube (ODC), accessed via STAC

The example below accesses a small stack of NRB data from Digital Earth Australia's (DEA's) ODC

**Open a STAC client to the catalog and configure anonymous access to AWS S3**

In [ ]:
from pystac_client import Client
from odc.stac import configure_s3_access

catalog = "https://explorer.dev.dea.ga.gov.au/stac"

stac_client = Client.open(catalog)

configure_s3_access(
    cloud_defaults=True, 
    aws_unsigned=True,
)

**Define the temporal and spatial bounds for you data**

In [ ]:
from odc.geo import BoundingBox

aus_bbox = BoundingBox(
    left=147.07251,
    bottom=-42.22120,
    right=147.24274,
    top=-42.03035,
    crs="EPSG:4326"
)

aus_start_date = "2024-06-01"
aus_end_date = "2025-11-15"

**Query the STAC catalog**

In [ ]:
collections_query = ["ga_s1_nrb_iw_hh_hv_0"]
aus_date_query = f"{aus_start_date}/{aus_end_date}"
aus_bbox_query = aus_bbox.bbox

aus_items = stac_client.search(
    collections=collections_query,
    datetime=aus_date_query,
    bbox=aus_bbox_query
).item_collection()

print(f"Found {len(aus_items)} items")

**Load the data into an `xarray.Dataset`**

In [ ]:
from odc.stac import load

# Lazy load our filtered data
aus_ds = load(
    aus_items,
    crs="utm",
    resolution=20,
    intersects=aus_bbox.boundary(),
    bands=["VV", "VH", "mask"],
    groupby="solar_day",
    chunks={},
)

aus_ds

**Iterate through each date in the stack and generate the water maps**

In [ ]:
from pathlib import Path

from hydrosar.water_map import make_water_map
from tqdm.auto import tqdm

water_mask_dir = Path.cwd() / "water_masks"
water_mask_dir.mkdir(exist_ok=True)

for dt in tqdm(aus_ds.time):
    vv = aus_ds["VV"].sel(time=dt)
    vh = aus_ds["VH"].sel(time=dt)

    date = "".join(str(dt.values.astype("datetime64[D]")).split("-"))
    water_extent_output = water_mask_dir / f'water_extent_{date}.tif'

    make_water_map(
        water_extent_output, 
        vv, 
        vh, 
        tile_shape=(100, 100),
        max_vv_threshold=-15.5, 
        max_vh_threshold=-23., 
        hand_threshold=15., 
        hand_fraction=0.8
    )

*HYDRO30_ODC_Stac_Processing.ipynb - Version 0.0.1 - Nov 2025*